# 06 — Concentration (geography, HHI, high-LVR)

**Plain English:** Where is the risk *concentrated*? We slice exposure and
default by geography (state) and by vintage, add the **HHI** (a single
concentration number) for the state book, and a **high-LVR concentration** view
(exposure by original loan-to-value band). Concentration is a portfolio risk in
its own right (APS 220 para 35) — these tables feed both the appetite limits
(notebook 07) and the monitoring pack.

**One-line terms**
- **Concentration** — how exposure clusters in a few states / cohorts / risky products; a portfolio risk in itself.
- **Exposure (UPB)** — current unpaid principal balance, i.e. money still at risk.
- **HHI** — Herfindahl–Hirschman Index, Σ(share %)² on a 0–10,000 scale; <1500 low · 1500–2500 moderate · >2500 high.
- **High-LVR** — loans whose *original* loan-to-value was above 90% (a higher-risk product, APS 220 para 35).
- **APS 330-style** — laid out like an APRA APS 330 credit-risk disclosure table. *Format only — illustrative, not a regulatory submission.*

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
pd.set_option("display.width", 120); pd.set_option("display.max_columns", 30)
import monitor as m
print("monitor library loaded — vintages:", m.VINTAGES)
panel = pd.read_parquet(m.PROC_DIR / "panel.parquet")

In [ ]:
# Latest snapshot per loan for a point-in-time concentration view --------
latest = panel.sort_values(["loan_seq", "mob"]).groupby("loan_seq").tail(1)
active = latest[latest.zb_code.isna()].copy()  # still on book

by_state = (active.groupby("prop_state")
            .agg(loans=("loan_seq", "size"),
                 exposure_upb=("cur_upb", "sum"),
                 pct_90plus=("bucket", lambda s: round(100 * s.isin(["90+", "Default"]).mean(), 2)))
            .sort_values("exposure_upb", ascending=False))
by_state["exposure_share_pct"] = (100 * by_state.exposure_upb / by_state.exposure_upb.sum()).round(2)
by_state.head(12)

In [ ]:
# Concentration by vintage (APS 330-style layout) -----------------------
by_vintage = (active.groupby("vintage")
              .agg(loans=("loan_seq", "size"),
                   exposure_upb=("cur_upb", "sum"),
                   avg_credit_score=("credit_score", "mean"),
                   pct_stage2=("stage", lambda s: round(100 * (s == "Stage 2").mean(), 2)),
                   pct_stage3=("stage", lambda s: round(100 * (s == "Stage 3").mean(), 2)))
              .reset_index())
by_vintage["exposure_upb"] = by_vintage.exposure_upb.round(0)
by_vintage["avg_credit_score"] = by_vintage.avg_credit_score.round(0)
by_vintage.to_csv(m.OUT_TABLES / "06_concentration_vintage.csv", index=False)
by_state.round(2).to_csv(m.OUT_TABLES / "06_concentration_state.csv")
by_vintage

In [ ]:
# HHI of the state book + high-LVR concentration (APS 220 para 35) -------
# HHI: one number for "how concentrated is the geography?" (companion commercial
# monitor reports the same measure). Computed on exposure shares in percent.
state_share = 100 * active.groupby("prop_state")["cur_upb"].sum() / active["cur_upb"].sum()
geo_hhi = m.hhi(state_share.values)
hhi_tbl = pd.DataFrame([{
    "dimension": "state (geography)",
    "n_buckets": int(state_share.shape[0]),
    "top_share_pct": round(float(state_share.max()), 2),
    "HHI": round(geo_hhi, 0),
    "classification": m.hhi_class(geo_hhi),
}])
hhi_tbl.to_csv(m.OUT_TABLES / "06_concentration_hhi.csv", index=False)
hhi_tbl

In [ ]:
# High-LVR concentration: exposure share by ORIGINAL LVR band ------------
active = active.copy()
active["lvr_band"] = m.lvr_band(active["ltv"])
lvr = (active.groupby("lvr_band", observed=False)
       .agg(loans=("loan_seq", "size"),
            exposure_upb=("cur_upb", "sum"),
            pct_90plus=("bucket", lambda s: round(100 * s.isin(["90+", "Default"]).mean(), 2)))
       )
lvr["exposure_share_pct"] = (100 * lvr.exposure_upb / lvr.exposure_upb.sum()).round(2)
high_lvr_share = float(lvr.loc[lvr.index.isin([">95", "90-95"]), "exposure_share_pct"].sum())
print(f"High-LVR (original LVR > 90%) share of exposure: {high_lvr_share:.2f}%")
lvr.round(2).to_csv(m.OUT_TABLES / "06_concentration_lvr.csv")
lvr.round(2)